# Валидация v5: полные метки и два режима истории

Этот ноутбук подготавливает протокол следующего обучения. Он не обучает модели,
не выбирает параметры по скрытому тесту и не меняет отправленный answer.csv.
Входы — три предоставленных Parquet. Все функции находятся в этом ноутбуке.

Полные положительные метки фиксируются до очистки истории. Для нового текста
удаляем все его контексты; для известного текста удаляем только удержанные
контексты. Фактическую доступность истории измеряем, а не предполагаем.
`search_category` участвует только в идентичности исходного контекста, как в v4;
признаки scoring в этом ноутбуке не строятся.


In [1]:
from pathlib import Path
import gc, hashlib, html, json, re, sys, unicodedata
from types import MappingProxyType
ROOT = Path.cwd()
if not (ROOT / 'train.parquet').exists():
    raise FileNotFoundError('Запустите notebook из папки с тремя Parquet')
if (ROOT / '.inspection_deps').is_dir():
    sys.path.insert(0, str(ROOT / '.inspection_deps'))
import numpy as np
import pandas as pd
SEED = 260926
CONFIG = {'validation_texts': 1400, 'cold_item_fraction': .9, 'development_fraction': .5}
RANKER_CONFIG = {'audit_queries': 600, 'fresh_audit_queries': 600, 'training_queries': 4000}
QUERY_COLS = ['search_query', 'search_location_id', 'search_is_delivery_search',
              'search_infm_params_text', 'search_category']
CACHE = ROOT / 'artifacts' / 'validation-v5'
CACHE.mkdir(parents=True, exist_ok=True)
train = pd.read_parquet(ROOT / 'train.parquet', columns=QUERY_COLS +
                       ['item_id', 'item_location_id', 'item_microcat_id'])
queries = pd.read_parquet(ROOT / 'benchmark_queries.parquet')
items = pd.read_parquet(ROOT / 'benchmark_items.parquet', columns=['item_id'])
items = items.sort_values('item_id').reset_index(drop=True)
ITEM_IDS = items.item_id.to_numpy(dtype=str)
ITEM_TO_ROW = {value: row for row, value in enumerate(ITEM_IDS)}
assert items.item_id.is_unique and queries.query_id.is_unique
for frame in [train, queries]:
    for column in ['search_query', 'search_infm_params_text']:
        frame[column] = frame[column].fillna('').astype('string[pyarrow]')


In [2]:
def normalize_text(value):
    value = unicodedata.normalize("NFKC", html.unescape(str(value))).lower().replace("ё", "е")
    value = re.sub(r"<[^>]+>", " ", value)
    return re.sub(r"\s+", " ", value).strip()

for frame in [train, queries]:
    frame['query_norm'] = frame.search_query.map(normalize_text).astype('string[pyarrow]')
    frame['context_key'] = [hashlib.sha256(json.dumps([str(v) for v in row],
        ensure_ascii=False).encode()).hexdigest()[:24]
        for row in frame[QUERY_COLS].itertuples(index=False, name=None)]
def select_query_contexts(history, count, seed):
    eligible_pairs = history[history.item_id.isin(ITEM_TO_ROW)]
    local_rng = np.random.default_rng(seed)
    texts = np.sort(eligible_pairs.query_norm.unique().astype(str))
    chosen_texts = local_rng.choice(texts, min(count, len(texts)), replace=False)
    contexts = eligible_pairs[eligible_pairs.query_norm.isin(chosen_texts)].drop_duplicates("context_key")
    chosen = []
    for _, group in contexts.sort_values("context_key").groupby("query_norm", sort=True):
        chosen.append(group.iloc[int(local_rng.integers(len(group)))])
    return pd.DataFrame(chosen)[[*QUERY_COLS, "query_norm", "context_key"]].sort_values("context_key").reset_index(drop=True)

def query_labels(history, query_frame):
    pairs = history[history.context_key.isin(query_frame.context_key) & history.item_id.isin(ITEM_TO_ROW)]
    grouped = pairs.groupby("context_key").item_id.agg(lambda s: set(ITEM_TO_ROW[x] for x in s))
    return [grouped.get(key, set()) for key in query_frame.context_key]

def purge_history(history, query_frame, relevant, seed):
    positives = np.array(sorted({ITEM_IDS[i] for truth in relevant for i in truth}))
    local_rng = np.random.default_rng(seed)
    cold_ids = set(local_rng.choice(positives, int(CONFIG["cold_item_fraction"] * len(positives)), replace=False))
    clean = history[~history.query_norm.isin(query_frame.query_norm) & ~history.item_id.isin(cold_ids)].copy()
    assert not set(query_frame.query_norm) & set(clean.query_norm)
    assert not cold_ids & set(clean.item_id)
    return clean, cold_ids


## Сохранение прежних контекстов для сравнения

Следующие две ячейки воспроизводят историческую выборку v4. Вызовы старого
query_labels нужны только для воспроизведения прежнего разбиения и диагностики
потерянных меток. При дальнейшей подготовке используем исключительно полный gold.
Все ранее просмотренные 3400 контекстов считаем development; независимым новым
тестом эта выборка больше не является.


In [3]:
pairs = train.drop_duplicates(["context_key", "item_id"]).copy()
eligible = pairs[pairs.item_id.isin(ITEM_TO_ROW)].copy()
rng = np.random.default_rng(SEED)
text_pool = np.sort(eligible.query_norm.unique().astype(str))
selected_texts = rng.choice(text_pool, size=min(CONFIG["validation_texts"], len(text_pool)), replace=False)
selected = eligible[eligible.query_norm.isin(selected_texts)]
context_pool = selected[["query_norm", "context_key"]].drop_duplicates().sort_values(["query_norm", "context_key"])
chosen_keys = []
for _, group in context_pool.groupby("query_norm", sort=True):
    chosen_keys.append(group.context_key.iloc[int(rng.integers(len(group)))])
validation_pairs = selected[selected.context_key.isin(chosen_keys)]
validation = validation_pairs.drop_duplicates("context_key")[[*QUERY_COLS, "query_norm", "context_key"]].copy()
validation = validation.sort_values("context_key").reset_index(drop=True)
labels_by_key = validation_pairs.groupby("context_key").item_id.agg(lambda s: sorted(set(s)))
labels = [set(ITEM_TO_ROW[item_id] for item_id in labels_by_key[key]) for key in validation.context_key]
val_positive_ids = np.sort(validation_pairs.item_id.unique().astype(str))
cold_item_ids = set(rng.choice(val_positive_ids, size=int(CONFIG["cold_item_fraction"] * len(val_positive_ids)), replace=False))
fit_pairs = pairs[~pairs.query_norm.isin(selected_texts) & ~pairs.item_id.isin(cold_item_ids)].copy()
assert not set(validation.query_norm) & set(fit_pairs.query_norm)
assert not cold_item_ids & set(fit_pairs.item_id)
assert all(labels)
split_order = rng.permutation(len(validation))
dev_indices = np.sort(split_order[:int(len(validation) * CONFIG["development_fraction"])])
holdout_indices = np.sort(split_order[int(len(validation) * CONFIG["development_fraction"]):])
validation["split"] = "holdout"
validation.loc[dev_indices, "split"] = "development"
validation["relevant_item_ids"] = [" ".join(labels_by_key[key]) for key in validation.context_key]
validation.to_parquet(CACHE / "validation.parquet", index=False)
print("History pairs:", len(fit_pairs), "; validation queries:", len(validation))
print("Development:", len(dev_indices), "; holdout:", len(holdout_indices))
print("Mean positives:", round(np.mean([len(x) for x in labels]), 3))
print("Actual unseen positive item fraction:", round(1 - validation_pairs.item_id.isin(fit_pairs.item_id).mean(), 4))
# Полные тексты train больше не нужны; сохраняем компактную историю для обучения.
HISTORY_COLS = [*QUERY_COLS, "query_norm", "context_key", "item_id", "item_location_id", "item_microcat_id"]
history_all = pairs[HISTORY_COLS].copy()
history_fit = fit_pairs[HISTORY_COLS].copy()
del train, pairs, fit_pairs, eligible, selected, validation_pairs
gc.collect()


History pairs: 428634 ; validation queries: 1400
Development: 700 ; holdout: 700
Mean positives: 1.026
Actual unseen positive item fraction: 0.952


In [4]:
audit_queries = select_query_contexts(history_fit, RANKER_CONFIG["audit_queries"], SEED + 101)
audit_labels = query_labels(history_fit, audit_queries)
ranker_history, audit_cold_ids = purge_history(history_fit, audit_queries, audit_labels, SEED + 102)
original_training_queries = select_query_contexts(ranker_history, RANKER_CONFIG["training_queries"], SEED + 103)
# These 600 texts were not direct training queries in the already inspected v0.2
# model. Remove their labels from all history before fitting the new comparison.
fresh_source = ranker_history[~ranker_history.query_norm.isin(original_training_queries.query_norm)]
fresh_queries = select_query_contexts(fresh_source, RANKER_CONFIG["fresh_audit_queries"], SEED + 501)
fresh_labels = query_labels(ranker_history, fresh_queries)
ranker_history, fresh_cold_ids = purge_history(ranker_history, fresh_queries, fresh_labels, SEED + 502)
training_queries = select_query_contexts(ranker_history, RANKER_CONFIG["training_queries"], SEED + 103)
assert all(fresh_labels)
assert not set(fresh_queries.query_norm) & set(original_training_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(training_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(validation.query_norm)
assert not set(fresh_queries.query_norm) & set(audit_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(ranker_history.query_norm)
assert not fresh_cold_ids & set(ranker_history.item_id)
fresh_protocol = {"fresh_queries": len(fresh_queries), "development_queries": len(validation) + len(audit_queries),
    "training_queries": len(training_queries), "query_text_overlap": 0, "purged_item_overlap": 0,
    "old_training_query_text_overlap": 0, "purged_positive_items": len(fresh_cold_ids),
    "used_for_model_selection": False}
fresh_protocol["role"] = "historical_v0.3_slice_now_development"
fresh_protocol["used_for_model_selection"] = True
(CACHE / "prior_split_protocol.json").write_text(json.dumps(fresh_protocol, indent=2), encoding="utf-8")
del fresh_source



old_direct_texts = set(training_queries.query_norm) | set(original_training_queries.query_norm)
old_direct_ids = {ITEM_IDS[i] for truth in query_labels(ranker_history, training_queries) for i in truth}
overlap_texts = set(ranker_history.loc[ranker_history.item_id.isin(old_direct_ids), 'query_norm'])
audit_source = ranker_history[~ranker_history.query_norm.isin(old_direct_texts | overlap_texts)]
new_audit = select_query_contexts(audit_source, 800, SEED + 702)
old_new_audit_labels = query_labels(ranker_history, new_audit)
development = pd.concat([validation[[*QUERY_COLS, 'query_norm', 'context_key']],
                         audit_queries, fresh_queries, new_audit], ignore_index=True)
assert development.context_key.is_unique


## Независимые gold-метки и очищаемая история

Gold содержит все наблюдённые положительные объявления доступного корпуса.
Неизменяемая структура защищает от случайного удаления меток при purge истории.
Без времени событий мы не можем независимо проверить повторный выбор в полностью
совпадающем контексте; показываем его долю отдельно, не выдаём её за честный holdout.


In [5]:
"""Validation contracts for v5; the notebook embeds these functions verbatim.

Gold labels come from the complete interaction table. History is a separate,
mutable view: removing information from it must never remove evaluation labels.
"""
from types import MappingProxyType

import numpy as np
import pandas as pd


def freeze_gold(full_pairs, item_to_row):
    """Snapshot all observed positives in the searchable corpus, once.

    Both the mapping and its values are immutable. Corpus membership is the only
    label filter; geography and historical purges do not affect the gold set.
    """
    eligible = full_pairs.loc[full_pairs.item_id.isin(item_to_row),
                              ['context_key', 'item_id']].drop_duplicates()
    grouped = eligible.groupby('context_key', sort=True).item_id.agg(
        lambda values: frozenset(item_to_row[value] for value in values))
    return MappingProxyType(grouped.to_dict())


def labels_from_gold(gold, query_frame):
    """Return immutable labels in query order; never accept a history argument."""
    return [gold.get(key, frozenset()) for key in query_frame.context_key]


def history_for_queries(history, query_frame, truths, item_ids, mode,
                        cold_fraction, seed, excluded_contexts=(),
                        excluded_texts=(), excluded_items=()):
    """Build an isolated feature history for a specified evaluation regime.

    unseen_text removes every context of the held text. held_context removes
    own context interactions but permits other contexts of the same text.
    Without timestamps, this does not evaluate repeats of the identical context.
    """
    if mode not in {'unseen_text', 'held_context'}:
        raise ValueError(f'Unknown history mode: {mode}')
    if not 0 <= cold_fraction <= 1:
        raise ValueError('cold_fraction must be in [0, 1]')
    if len(query_frame) != len(truths):
        raise ValueError('One gold set is required per query')
    positive_ids = np.array(sorted({item_ids[i] for truth in truths for i in truth}),
                            dtype=str)
    random = np.random.default_rng(seed)
    cold = set(random.choice(positive_ids, int(cold_fraction * len(positive_ids)),
                             replace=False))
    blocked_contexts = set(query_frame.context_key) | set(excluded_contexts)
    blocked_texts = set(excluded_texts)
    if mode == 'unseen_text':
        blocked_texts.update(query_frame.query_norm)
    blocked_items = cold | set(excluded_items)
    fit = history.loc[~history.context_key.isin(blocked_contexts)
                      & ~history.query_norm.isin(blocked_texts)
                      & ~history.item_id.isin(blocked_items)].copy()
    assert not blocked_contexts & set(fit.context_key)
    assert not blocked_items & set(fit.item_id)
    assert not blocked_texts & set(fit.query_norm)
    known = query_frame.query_norm.isin(fit.query_norm).to_numpy(dtype=bool)
    if mode == 'unseen_text':
        assert not known.any()
    return fit, cold, known


def oof_assignments(query_frame, mode, folds=3, seed=260926):
    """Group by text for cold queries, by context for held-context queries.

    A held context includes all its positives and duplicate rows. Different
    contexts of one text may appear in the fit history in the second regime.
    """
    if mode not in {'unseen_text', 'held_context'}:
        raise ValueError(f'Unknown history mode: {mode}')
    if folds < 2:
        raise ValueError('At least two folds are required')
    column = 'query_norm' if mode == 'unseen_text' else 'context_key'
    values = np.sort(query_frame[column].unique().astype(str))
    random = np.random.default_rng(seed)
    shuffled = values[random.permutation(len(values))]
    mapping = {key: position % folds for position, key in enumerate(shuffled)}
    return query_frame[column].map(mapping).to_numpy(dtype=np.int32)


def history_coverage(query_frame, truths, fit, item_ids):
    """Measure actual availability rather than infer it from a regime's name."""
    known = query_frame.query_norm.isin(fit.query_norm).to_numpy(dtype=bool)
    observed_items = set(fit.item_id)
    fractions = [sum(item_ids[i] in observed_items for i in truth) / len(truth)
                 for truth in truths if truth]
    return {'contexts': len(query_frame), 'positive_pairs': sum(map(len, truths)),
            'known_text_fraction': float(known.mean()) if len(known) else None,
            'positive_item_seen_macro_fraction': float(np.mean(fractions)) if fractions else None,
            'own_context_overlap': len(set(query_frame.context_key) & set(fit.context_key))}


gold = freeze_gold(history_all, ITEM_TO_ROW)
development_truths = labels_from_gold(gold, development)
assert all(development_truths)
restored_new = labels_from_gold(gold, new_audit)
label_changes = {'old_new_audit_positives': sum(map(len, old_new_audit_labels)),
    'full_new_audit_positives': sum(map(len, restored_new)),
    'affected_new_audit_contexts': sum(a != b for a, b in zip(old_new_audit_labels, restored_new))}
assert label_changes == {'old_new_audit_positives': 815, 'full_new_audit_positives': 822,
                         'affected_new_audit_contexts': 6}
known_text = queries.query_norm.isin(history_all.query_norm)
known_context = queries.context_key.isin(history_all.context_key)
target = {'unseen_text_fraction': float((~known_text).mean()),
    'known_text_new_context_fraction': float((known_text & ~known_context).mean()),
    'exact_context_fraction': float(known_context.mean()),
    'exact_context_evaluation_limitation': 'No timestamps or independent repeated events'}
assert np.isclose(sum(target[k] for k in ['unseen_text_fraction',
    'known_text_new_context_fraction', 'exact_context_fraction']), 1)
print('Исправление меток:', json.dumps(label_changes, ensure_ascii=False))
print('Режимы бенчмарка:', json.dumps(target, ensure_ascii=False))
profiles = []
for mode in ['unseen_text', 'held_context']:
    for fraction in [0., .5, .9, 1.]:
        fit, cold, _ = history_for_queries(history_all, development,
            development_truths, ITEM_IDS, mode, fraction, SEED + 801)
        profile = history_coverage(development, development_truths, fit, ITEM_IDS)
        profile.update(mode=mode, cold_fraction=fraction, cold_items=len(cold))
        profiles.append(profile)
        assert profile['own_context_overlap'] == 0
        del fit
print(pd.DataFrame(profiles).to_string(index=False))


Исправление меток: {"old_new_audit_positives": 815, "full_new_audit_positives": 822, "affected_new_audit_contexts": 6}
Режимы бенчмарка: {"unseen_text_fraction": 0.6264274061990212, "known_text_new_context_fraction": 0.3295269168026101, "exact_context_fraction": 0.04404567699836868, "exact_context_evaluation_limitation": "No timestamps or independent repeated events"}
 contexts  positive_pairs  known_text_fraction  positive_item_seen_macro_fraction  own_context_overlap         mode  cold_fraction  cold_items
     3400            3480             0.000000                           0.456000                    0  unseen_text            0.0           0
     3400            3480             0.000000                           0.232951                    0  unseen_text            0.5        1683
     3400            3480             0.000000                           0.045539                    0  unseen_text            0.9        3029
     3400            3480             0.000000           

## Подготовка следующего обучающего набора

Контексты для прямого обучения выбираем по полным gold-меткам, а не по очищенной
истории. Иначе удаление item из вспомогательной истории теряет ещё и обучающий пример.
Тексты всех development-запросов исключаем из нового прямого обучения.
История обучения также исключает эти тексты и выбранные cold items.

Для каждого контекста предусмотрены два режима OOF. В одном группируем по тексту,
в другом по полному контексту. Второй режим может фактически стать неизвестным
после очистки: это отмечено отдельным флагом known_text, необходимым при обучении.
Ранги и hard negatives будут рассчитываться следующим этапом по полному пулу.


In [6]:
def context_sample(history, count, seed, include=None):
    eligible = history[history.item_id.isin(ITEM_TO_ROW)].drop_duplicates('context_key')
    local = np.random.default_rng(seed)
    eligible = eligible.sort_values('context_key').iloc[local.permutation(len(eligible))]
    bounded = eligible.groupby('query_norm', sort=False).head(EXPERIMENT_CONFIG['contexts_per_text'])
    if include is not None:
        bounded = pd.concat([include, bounded], ignore_index=True).drop_duplicates('context_key')
    return bounded.head(count)[[*QUERY_COLS,'query_norm','context_key']].sort_values('context_key').reset_index(drop=True)

# Reserve text groups not used for direct supervised fitting in v4. Reconstruct
# that list locally rather than depend on an untracked cached parquet file.
EXPERIMENT_CONFIG = {'contexts_per_text': 4}
old_clean, _ = purge_history(ranker_history, new_audit, old_new_audit_labels, SEED + 703)
old_expanded = context_sample(old_clean, 20000, SEED + 704, training_queries)
previous_direct_texts = set(old_expanded.query_norm) | set(development.query_norm)
control_source = history_all[~history_all.query_norm.isin(previous_direct_texts)]
held_control = select_query_contexts(control_source, 600, SEED + 805)
held_control_truths = labels_from_gold(gold, held_control)
assert len(held_control) == 600 and all(held_control_truths)
assert not set(held_control.query_norm) & previous_direct_texts
evaluation_contexts = pd.concat([development, held_control], ignore_index=True)
evaluation_truths = development_truths + held_control_truths
training_history, development_cold, _ = history_for_queries(history_all,
    evaluation_contexts, evaluation_truths, ITEM_IDS, 'unseen_text', .9, SEED + 802)
candidate_contexts = history_all[history_all.context_key.isin(gold)].drop_duplicates('context_key')
candidate_contexts = candidate_contexts[~candidate_contexts.query_norm.isin(evaluation_contexts.query_norm)]
candidate_contexts = candidate_contexts[[*QUERY_COLS, 'query_norm', 'context_key']].sort_values('context_key').reset_index(drop=True)
candidate_truths = labels_from_gold(gold, candidate_contexts)
assert all(candidate_truths)
assert not set(candidate_contexts.query_norm) & set(evaluation_contexts.query_norm)
fold_profiles, assignment_rows = [], []
for mode in ['unseen_text', 'held_context']:
    assignments = oof_assignments(candidate_contexts, mode, 3, SEED + 803)
    for fold in range(3):
        positions = np.flatnonzero(assignments == fold)
        q = candidate_contexts.iloc[positions]
        truth = [candidate_truths[i] for i in positions]
        fit, cold, actual_known = history_for_queries(training_history, q, truth,
            ITEM_IDS, mode, .9, SEED + 804 + fold)
        profile = history_coverage(q, truth, fit, ITEM_IDS)
        profile.update(mode=mode, fold=fold, cold_items=len(cold), history_pairs=len(fit))
        fold_profiles.append(profile)
        assignment_rows.extend({'context_key': key, 'mode': mode, 'fold': fold,
            'actual_known_text': bool(known)} for key, known in zip(q.context_key, actual_known))
        assert profile['own_context_overlap'] == 0
        if mode == 'unseen_text':
            assert profile['known_text_fraction'] == 0
        del fit
        gc.collect()
prepared = candidate_contexts.copy()
prepared['relevant_item_ids'] = [' '.join(ITEM_IDS[i] for i in sorted(truth)) for truth in candidate_truths]
prepared.to_parquet(CACHE / 'training_contexts.parquet', index=False)
prepared_dev = development.copy()
prepared_dev['relevant_item_ids'] = [' '.join(ITEM_IDS[i] for i in sorted(truth)) for truth in development_truths]
prepared_dev.to_parquet(CACHE / 'development_contexts.parquet', index=False)
prepared_control = held_control.copy()
prepared_control['relevant_item_ids'] = [' '.join(ITEM_IDS[i] for i in sorted(truth)) for truth in held_control_truths]
prepared_control.to_parquet(CACHE / 'held_control_contexts.parquet', index=False)
pd.DataFrame(assignment_rows).to_parquet(CACHE / 'oof_assignments.parquet', index=False)
pd.DataFrame(profiles).to_csv(CACHE / 'history_regimes.csv', index=False)
pd.DataFrame(fold_profiles).to_csv(CACHE / 'oof_history_audit.csv', index=False)
report = {'stage': 'validation-v5', 'gold_source': 'complete unpurged train, corpus positives only',
    'gold_contexts': len(gold), 'gold_positive_pairs': sum(map(len, gold.values())),
    'label_correction': label_changes, 'benchmark_regimes': target,
    'development_contexts': len(development), 'development_role': 'previously viewed development',
    'held_control_contexts': len(held_control), 'held_control_direct_text_overlap': 0,
    'held_control_limitation': 'Frozen v4 priors may have used these interactions in auxiliary history; strict validation requires refitting affected priors',
    'training_contexts': len(candidate_contexts),
    'training_texts': int(candidate_contexts.query_norm.nunique()),
    'training_positives': sum(map(len, candidate_truths)),
    'oof_audits': fold_profiles, 'evaluation_history_profiles': profiles,
    'new_models_fitted': False, 'answer_csv_modified': False,
    'next_stage': 'full-pool retrieval, stronger hard negatives, mixed-regime ranker training'}
(CACHE / 'protocol.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k: v for k, v in report.items() if k not in ['oof_audits', 'evaluation_history_profiles']},
                 ensure_ascii=False, indent=2))
print(pd.DataFrame(fold_profiles).to_string(index=False))


{
  "stage": "validation-v5",
  "gold_source": "complete unpurged train, corpus positives only",
  "gold_contexts": 26556,
  "gold_positive_pairs": 29572,
  "label_correction": {
    "old_new_audit_positives": 815,
    "full_new_audit_positives": 822,
    "affected_new_audit_contexts": 6
  },
  "benchmark_regimes": {
    "unseen_text_fraction": 0.6264274061990212,
    "known_text_new_context_fraction": 0.3295269168026101,
    "exact_context_fraction": 0.04404567699836868,
    "exact_context_evaluation_limitation": "No timestamps or independent repeated events"
  },
  "development_contexts": 3400,
  "development_role": "previously viewed development",
  "held_control_contexts": 600,
  "held_control_direct_text_overlap": 0,
  "held_control_limitation": "Frozen v4 priors may have used these interactions in auxiliary history; strict validation requires refitting affected priors",
  "training_contexts": 19031,
  "training_texts": 8174,
  "training_positives": 21480,
  "new_models_fitted": f